# SEER Breast Cancer 2010–2017 — Preprocessing (v4)

**Study:** factorial benchmark of censoring handling (Zero / Discard / IPCW) × imbalance handling (none / SMOTE) for classifying breast-cancer death by horizon τ ∈ {36, 60, 120} months. See `SEER_BC_Revision_Spec.md`.

**Data:** SEER 17, November 2025 submission (Session 6 extract: 392,728 records, ages 20–90+, follow-up to 31 Dec 2023).

**Steps**
1. Load the extract and check its schema.
2. Apply the eligibility cascade (**one primary only**, complete key covariates) and log every step.
3. Build features: harmonized stage/TNM, and coded counts split into number + description.
4. Code the outcome (censored / breast-cancer death / other-cause death) and the per-horizon labels.
5. Validate the outcome (coding checks, Kaplan–Meier, Aalen–Johansen) and build tables T1 and S3.
6. Check that diagnosis era cannot be predicted from the features.
7. Save `model_df.csv`, `exclusion_log.csv`, the T1/S3 tables and `metadata.json`.

Nothing the models use is fitted here. Scaling, imputation, feature selection and the censoring model Ĝ are all fitted inside each CV training fold.

In [ ]:
from pathlib import Path
import config
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
HORIZONS = [36, 60, 120]  # months (spec §4.3)

DATA_PATH = config.SEER_EXTRACT   # SEER*Stat case listing (config.py)

NOTEBOOK_DIR = config.WORK_DIR    # all outputs below this folder (config.py)
OUTPUT_DIR = NOTEBOOK_DIR / "outputs_model_ready"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {len(df.columns):,}")
df.head()

### Schema check

Stops early on the wrong file: every column used below must exist. The row count should match the Session 6 printout (392,728); a mismatch only warns, because a SEER data correction can shift it slightly.

The extract has no SEER registry field, so the censoring model uses year of diagnosis and age only.

In [ ]:
EXPECTED_N_RAW = 392_728  # Session 6 printout: "Number of Case Records Used For Analysis"

REQUIRED_COLUMNS = [
    "Age recode with single ages and 90+",
    "Age recode with <1 year olds and 90+",
    "Patient ID",
    "Sequence number",
    "Type of Reporting Source",
    "Total number of in situ/malignant tumors for patient",
    "Median household income inflation adj to 2024",
    "Rural-Urban Continuum Code",
    "Year of diagnosis",
    "Marital status at diagnosis",
    "ER Status Recode Breast Cancer (1990+)",
    "PR Status Recode Breast Cancer (1990+)",
    "Derived HER2 Recode (2010+)",
    "Breast Subtype (2010+)",
    "Laterality",
    "Grade Recode (thru 2017)",
    "Histologic Type ICD-O-3",
    "Histology recode - broad groupings",
    "ICD-O-3 Hist/behav, malignant",
    "Primary Site",
    "Primary Site - labeled",
    "Tumor Size Over Time Recode (1988+)",
    "Regional nodes examined (1988+)",
    "Regional nodes positive (1988+)",
    "LN Positive Axillary Level I-II Recode (2010+)",
    "Derived AJCC Stage Group, 7th ed (2010-2015)",
    "Derived AJCC T, 7th ed (2010-2015)",
    "Derived AJCC N, 7th ed (2010-2015)",
    "Derived AJCC M, 7th ed (2010-2015)",
    "7th Edition Stage Group Recode (2016-2017)",
    "Derived SEER Cmb Stg Grp (2016-2017)",
    "Derived SEER Combined T (2016-2017)",
    "Derived SEER Combined N (2016-2017)",
    "Derived SEER Combined M (2016-2017)",
    "SEER Combined Mets at DX-bone (2010+)",
    "SEER Combined Mets at DX-brain (2010+)",
    "SEER Combined Mets at DX-liver (2010+)",
    "SEER Combined Mets at DX-lung (2010+)",
    "RX Summ--Surg Prim Site (1998-2022)",
    "RX Summ--Surg/Rad Seq",
    "Radiation recode",
    "Chemotherapy recode (yes, no/unk)",
    "RX Summ--Systemic/Sur Seq (2007+)",
    "Time from diagnosis to treatment in days recode",
    "SEER cause-specific death classification",
    "SEER other cause of death classification",
    "Survival months",
    "Survival Days",
    "Survival months flag",
    "Vital status recode (study cutoff used)",
]

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
assert not missing_columns, "Missing required columns:\n  - " + "\n  - ".join(missing_columns)

if abs(len(df) - EXPECTED_N_RAW) > 500:
    print(f"WARNING: loaded {len(df):,} rows, expected {EXPECTED_N_RAW:,}. Check DATA_PATH.")
else:
    print(f"Schema check passed: {len(df):,} rows, all {len(REQUIRED_COLUMNS)} required columns present.")

## Data investigation

In [ ]:
def show_counts(columns):
    """Print the number of levels and the value counts of each column."""
    for col in columns:
        counts = df[col].value_counts(dropna=False)
        print(f"=== {col} ({len(counts)} levels) ===")
        display(counts)


show_counts([
    "Survival months flag",
    "Vital status recode (study cutoff used)",
    "SEER cause-specific death classification",
    "SEER other cause of death classification",
    "Breast Subtype (2010+)",
    "Tumor Size Over Time Recode (1988+)",
    "Regional nodes examined (1988+)",
    "Regional nodes positive (1988+)",
])

### Columns added in the Session 6 extract

The eligibility masks and parsers below match these exact label strings. If a label looks different from what the code expects, fix it here.

In [ ]:
show_counts([
    "Sequence number",
    "Type of Reporting Source",
    "Total number of in situ/malignant tumors for patient",
    "Median household income inflation adj to 2024",
    "Rural-Urban Continuum Code",
])

age_single = df["Age recode with single ages and 90+"].value_counts().sort_index()
print(f"Distinct single-age labels: {len(age_single)}")
display(pd.concat([age_single.head(5), age_single.tail(3)]))

### Stage/TNM labels by era

In [ ]:
show_counts([
    "Derived AJCC Stage Group, 7th ed (2010-2015)",
    "Derived AJCC T, 7th ed (2010-2015)",
    "Derived AJCC N, 7th ed (2010-2015)",
    "Derived AJCC M, 7th ed (2010-2015)",
    "7th Edition Stage Group Recode (2016-2017)",
    "Derived SEER Cmb Stg Grp (2016-2017)",
    "Derived SEER Combined T (2016-2017)",
    "Derived SEER Combined N (2016-2017)",
    "Derived SEER Combined M (2016-2017)",
])

## Cohort filtering helpers

In [ ]:
APPLY_TREATMENT_EXCLUSION = False  # True: drop records with uncertain radiotherapy / systemic sequence
EXCLUDE_CHEMO_NO_UNKNOWN = False   # True: also treat "Chemotherapy No/Unknown" as uncertain

UNKNOWN_TEXT_TOKENS = (
    "unknown",
    "blank",
    "not documented",
    "not assessed",
    "not applicable",
    "unspecified",
    "recode not available",
)


def has_unknown_text(series: pd.Series, extra_tokens=()) -> pd.Series:
    tokens = tuple(UNKNOWN_TEXT_TOKENS) + tuple(extra_tokens)
    lower = series.fillna("").astype(str).str.lower()
    return lower.apply(lambda value: any(token in value for token in tokens))


def is_valid_receptor(series: pd.Series) -> pd.Series:
    return series.isin(["Positive", "Negative"])


def is_valid_mets_site(series: pd.Series) -> pd.Series:
    return series.isin(["Yes", "No"])


def is_known_stage(series: pd.Series) -> pd.Series:
    bad = {"", "Blank(s)", "NA", "UNK Stage", "99", "Not applicable"}
    return ~series.isin(bad) & ~has_unknown_text(series)


def is_known_tnm(series: pd.Series) -> pd.Series:
    bad = {"", "Blank(s)", "NA", "Not applicable"}
    upper = series.fillna("").astype(str).str.upper()
    return ~series.isin(bad) & ~has_unknown_text(series) & ~upper.str.endswith("X")


def is_known_node_count(series: pd.Series) -> pd.Series:
    """Two-digit node code other than 98 (not examined) / 99 (unknown)."""
    return series.str.fullmatch(r"\d{2}").fillna(False) & ~series.isin(["98", "99"])


def log_step(name: str, data: pd.DataFrame, keep_mask: pd.Series, log: list[dict]) -> pd.DataFrame:
    """Keep rows where keep_mask is True and record the step in the exclusion log."""
    keep_mask = keep_mask.reindex(data.index, fill_value=False)
    before = len(data)
    kept = int(keep_mask.sum())
    log.append({"step": name, "rows_before": before, "rows_excluded": before - kept, "rows_after": kept})
    print(f"{name}: excluded {before - kept:,}; remaining {kept:,}")
    return data.loc[keep_mask].copy()


def log_retained(name: str, data: pd.DataFrame, log: list[dict]) -> None:
    """Record a step that keeps every row (handled by recoding, not exclusion)."""
    log.append({"step": name, "rows_before": len(data), "rows_excluded": 0, "rows_after": len(data)})
    print(f"{name}: excluded 0; remaining {len(data):,}")


def find_column(patterns, required=True):
    for pattern in patterns:
        hit = next((col for col in df.columns if re.search(pattern, str(col), re.I)), None)
        if hit is not None:
            return hit
    if required:
        raise KeyError(f"Could not resolve column from patterns: {patterns}")
    return None


RACE_COL = find_column([r"race and origin recode", r"race recode"])
print(f"Race column: {RACE_COL}")

## Eligibility cascade

Each step is logged in `exclusion_log`, which is the source of the flow diagram. Steps that keep all records but recode them are logged with zero exclusions.

| Step | Rule | Reason |
|---|---|---|
| Reporting source | Drop death-certificate-only and autopsy-only cases | No follow-up interval |
| Sequence number | Keep **One primary only** | Study cohort definition (see note) |
| Demographics, receptors, laterality, TNM, metastatic sites, grade/histology | Drop unknown values | Eligibility-defining covariates |
| In situ (Tis / stage 0) | Drop | Outside an invasive-cancer estimand; also coded differently in 2010–15 and 2016–17, which would reveal the era |
| Tumor size, axillary node counts | **Keep**; special/unknown codes → `NaN` + descriptive category | Excluding unknown node counts removed older women disproportionately |
| Treatment uncertainty | Keep as flags | Optional exclusion is switched off |
| Unknown cause of death | Drop | Outcome cannot be labelled; tabulated by age for the supplement |
| Follow-up | Drop unknown or zero survival time | No usable follow-up |

**Note on "One primary only".** SEER updates the sequence number when a new tumour is diagnosed, so this status is known only at the end of follow-up. Patients who later develop a second cancer are excluded, and longer survivors have more time to develop one. This removes some other-cause deaths (from later cancers) and may slightly lower other-cause incidence. State it as a limitation in Methods.

In [ ]:
exclusion_log = []
cohort = df.copy()

year = pd.to_numeric(cohort["Year of diagnosis"], errors="coerce")
cohort = log_step("Study period: diagnosis year 2010-2017", cohort, year.between(2010, 2017), exclusion_log)

# Reporting source: case-insensitive match so a wording change cannot disable the filter
reporting = cohort["Type of Reporting Source"].str.lower()
reporting_mask = ~(
    reporting.str.contains("death certificate", regex=False)
    | reporting.str.contains("autopsy", regex=False)
)
cohort = log_step("Exclude death-certificate-only and autopsy-only cases", cohort, reporting_mask, exclusion_log)

# Sequence number: the extract should hold first primaries only
seq = cohort["Sequence number"]
unexpected_seq = set(seq.unique()) - {"One primary only", "1st of 2 or more primaries"}
assert not unexpected_seq, f"Unexpected Sequence number labels: {unexpected_seq}. Check the SEER*Stat selection."
cohort = log_step("Restrict to one primary only", cohort, seq.eq("One primary only"), exclusion_log)

# One record per patient; duplicates could put the same patient in train and test folds
dup_ids = cohort.loc[cohort["Patient ID"].duplicated(keep=False), "Patient ID"]
assert dup_ids.empty, f"{dup_ids.nunique():,} Patient IDs appear more than once"

# "N/A" death labels would mean a non-first tumour slipped through
for col in ["SEER cause-specific death classification", "SEER other cause of death classification"]:
    assert not cohort[col].str.contains("n/a", case=False, regex=False).any(), f"{col} has N/A labels"

# Snapshot before covariate exclusions (used by the missingness audit)
df_eligible = cohort.copy()

demographic_mask = (
    cohort["Age recode with <1 year olds and 90+"].ne("")
    & cohort["Year of diagnosis"].ne("")
    & ~cohort[RACE_COL].isin(["", "Unknown", "Non-Hispanic Unknown Race"])
    & ~cohort["Marital status at diagnosis"].isin(["", "Unknown"])
)
cohort = log_step("Exclude missing demographic information", cohort, demographic_mask, exclusion_log)

receptor_mask = (
    is_valid_receptor(cohort["ER Status Recode Breast Cancer (1990+)"])
    & is_valid_receptor(cohort["PR Status Recode Breast Cancer (1990+)"])
    & is_valid_receptor(cohort["Derived HER2 Recode (2010+)"])
)
cohort = log_step("Exclude unknown/borderline ER, PR, or HER2 status", cohort, receptor_mask, exclusion_log)

laterality_mask = cohort["Laterality"].isin(["Left - origin of primary", "Right - origin of primary"])
cohort = log_step("Exclude unknown laterality and bilateral breast cancer", cohort, laterality_mask, exclusion_log)

# TNM: each era is checked against its own columns
year = pd.to_numeric(cohort["Year of diagnosis"], errors="coerce")
known_2010_2015_tnm = (
    is_known_stage(cohort["Derived AJCC Stage Group, 7th ed (2010-2015)"])
    & is_known_tnm(cohort["Derived AJCC T, 7th ed (2010-2015)"])
    & is_known_tnm(cohort["Derived AJCC N, 7th ed (2010-2015)"])
    & is_known_tnm(cohort["Derived AJCC M, 7th ed (2010-2015)"])
)
known_2016_2017_tnm = (
    (
        is_known_stage(cohort["7th Edition Stage Group Recode (2016-2017)"])
        | is_known_stage(cohort["Derived SEER Cmb Stg Grp (2016-2017)"])
    )
    & is_known_tnm(cohort["Derived SEER Combined T (2016-2017)"])
    & is_known_tnm(cohort["Derived SEER Combined N (2016-2017)"])
    & is_known_tnm(cohort["Derived SEER Combined M (2016-2017)"])
)
tnm_mask = (year.between(2010, 2015) & known_2010_2015_tnm) | (year.between(2016, 2017) & known_2016_2017_tnm)
cohort = log_step("Exclude unspecified TNM stage", cohort, tnm_mask, exclusion_log)

# In situ (Tis / stage 0), from the raw era-specific columns.
# Era masks are recomputed because the index shrank in the previous step.
year = pd.to_numeric(cohort["Year of diagnosis"], errors="coerce")
tis_1015 = (
    cohort["Derived AJCC Stage Group, 7th ed (2010-2015)"].eq("0")
    | cohort["Derived AJCC T, 7th ed (2010-2015)"].eq("Tis")
)
t_1617 = cohort["Derived SEER Combined T (2016-2017)"].str.replace(r"^[cp]", "", regex=True).str.upper()
tis_1617 = (
    cohort["7th Edition Stage Group Recode (2016-2017)"].eq("0")
    | cohort["Derived SEER Cmb Stg Grp (2016-2017)"].eq("0")
    | t_1617.eq("IS")
)
invasive_mask = ~((year.between(2010, 2015) & tis_1015) | (year.between(2016, 2017) & tis_1617))
cohort = log_step("Exclude in situ presentations (Tis / stage 0, no invasive tumor)", cohort, invasive_mask, exclusion_log)

log_retained(
    "Tumor size: all records retained; special codes recoded to NaN + descriptive category",
    cohort,
    exclusion_log,
)

mets_cols = [
    "SEER Combined Mets at DX-bone (2010+)",
    "SEER Combined Mets at DX-brain (2010+)",
    "SEER Combined Mets at DX-liver (2010+)",
    "SEER Combined Mets at DX-lung (2010+)",
]
mets_mask = pd.concat([is_valid_mets_site(cohort[col]) for col in mets_cols], axis=1).all(axis=1)
cohort = log_step("Exclude unknown metastatic-site values", cohort, mets_mask, exclusion_log)

n_axillary_unknown = int(
    (
        ~(
            is_known_node_count(cohort["Regional nodes examined (1988+)"])
            & is_known_node_count(cohort["Regional nodes positive (1988+)"])
        )
    ).sum()
)
print(f"Records with unknown/procedure node codes (kept): {n_axillary_unknown:,}")
log_retained(
    "Axillary node status: all records retained; unknown/procedure codes recoded to NaN + descriptive category",
    cohort,
    exclusion_log,
)

cohort["radiotherapy_uncertain"] = cohort["Radiation recode"].isin(
    ["None/Unknown", "Recommended, unknown if administered"]
) | cohort["RX Summ--Surg/Rad Seq"].isin(
    [
        "No radiation and/or no surgery; unknown if surgery and/or radiation given",
        "Sequence unknown, but both were given",
    ]
)
cohort["systemic_sequence_unknown"] = cohort["RX Summ--Systemic/Sur Seq (2007+)"].isin(["Sequence unknown"])
cohort["chemo_no_unknown"] = cohort["Chemotherapy recode (yes, no/unk)"].isin(["No/Unknown"])

treatment_uncertainty_components = ["radiotherapy_uncertain", "systemic_sequence_unknown"]
if EXCLUDE_CHEMO_NO_UNKNOWN:
    treatment_uncertainty_components.append("chemo_no_unknown")
cohort["treatment_uncertain_any"] = cohort[treatment_uncertainty_components].any(axis=1)

if APPLY_TREATMENT_EXCLUSION:
    cohort = log_step(
        "Exclude uncertain systemic treatment or radiotherapy",
        cohort,
        ~cohort["treatment_uncertain_any"],
        exclusion_log,
    )
else:
    log_retained("Treatment uncertainty retained as flags; optional exclusion not applied", cohort, exclusion_log)

histology_mask = (
    ~cohort["Grade Recode (thru 2017)"].isin(["", "Unknown"])
    & ~cohort["Histology recode - broad groupings"].isin(["8000-8009: unspecified neoplasms"])
)
cohort = log_step("Exclude unknown histologic grade and type", cohort, histology_mask, exclusion_log)

# Unknown cause of death: check both classification columns; tabulate by age before excluding
is_dead = cohort["Vital status recode (study cutoff used)"].eq("Dead")
unknown_cod = (
    cohort["SEER cause-specific death classification"].eq("Dead (missing/unknown COD)")
    | cohort["SEER other cause of death classification"].eq("Dead (missing/unknown COD)")
)
print("Unknown cause of death among deaths, by age band (supplement):")
ucod_tab = pd.crosstab(
    cohort.loc[is_dead, "Age recode with <1 year olds and 90+"],
    unknown_cod.loc[is_dead].map({True: "unknown COD", False: "known COD"}),
)
if "unknown COD" not in ucod_tab.columns:
    ucod_tab["unknown COD"] = 0
ucod_tab["pct_unknown"] = (ucod_tab["unknown COD"] / ucod_tab.sum(axis=1) * 100).round(2)
display(ucod_tab)
cohort = log_step("Exclude deaths with missing/unknown cause of death", cohort, ~unknown_cod, exclusion_log)

followup_mask = (
    cohort["Survival months"].ne("Unknown")
    & cohort["Survival months"].ne("0000")
    & cohort["Survival Days"].ne("Unknown")
    & cohort["Survival months flag"].eq("Complete dates are available and there are more than 0 days of survival")
)
cohort = log_step("Exclude incomplete follow-up (unknown or zero survival time)", cohort, followup_mask, exclusion_log)

print(cohort.shape)
pd.DataFrame(exclusion_log)

### Missingness by age band (before covariate exclusions)

Measurement only; nothing is excluded here. The first table shows how often each covariate is unknown per age band in the eligible cohort. The second shows the share of each band that survives the cascade. Older bands lose more records, so report both tables in the supplement.

In [ ]:
band_col = "Age recode with <1 year olds and 90+"
year_eligible = pd.to_numeric(df_eligible["Year of diagnosis"], errors="coerce")

audit_masks = {
    "ER/PR/HER2 not Positive/Negative": ~(
        is_valid_receptor(df_eligible["ER Status Recode Breast Cancer (1990+)"])
        & is_valid_receptor(df_eligible["PR Status Recode Breast Cancer (1990+)"])
        & is_valid_receptor(df_eligible["Derived HER2 Recode (2010+)"])
    ),
    "Grade unknown": df_eligible["Grade Recode (thru 2017)"].isin(["", "Unknown"]),
    "TNM/stage unknown (era-specific)": ~(
        (
            year_eligible.between(2010, 2015)
            & is_known_stage(df_eligible["Derived AJCC Stage Group, 7th ed (2010-2015)"])
            & is_known_tnm(df_eligible["Derived AJCC T, 7th ed (2010-2015)"])
            & is_known_tnm(df_eligible["Derived AJCC N, 7th ed (2010-2015)"])
            & is_known_tnm(df_eligible["Derived AJCC M, 7th ed (2010-2015)"])
        )
        | (
            year_eligible.between(2016, 2017)
            & (
                is_known_stage(df_eligible["7th Edition Stage Group Recode (2016-2017)"])
                | is_known_stage(df_eligible["Derived SEER Cmb Stg Grp (2016-2017)"])
            )
            & is_known_tnm(df_eligible["Derived SEER Combined T (2016-2017)"])
            & is_known_tnm(df_eligible["Derived SEER Combined N (2016-2017)"])
            & is_known_tnm(df_eligible["Derived SEER Combined M (2016-2017)"])
        )
    ),
    "Axillary node counts unknown (retained)": ~(
        is_known_node_count(df_eligible["Regional nodes examined (1988+)"])
        & is_known_node_count(df_eligible["Regional nodes positive (1988+)"])
    ),
    "Tumor size not an exact measurement": ~pd.to_numeric(
        df_eligible["Tumor Size Over Time Recode (1988+)"].str.extract(r"^\s*(\d+)", expand=False),
        errors="coerce",
    ).between(1, 400),
}

miss_tab = pd.DataFrame(
    {name: mask.groupby(df_eligible[band_col]).mean() * 100 for name, mask in audit_masks.items()}
).round(2)

# Unknown COD: denominator is deaths only
dead_eligible = df_eligible["Vital status recode (study cutoff used)"].eq("Dead")
ucod_eligible = (
    df_eligible["SEER cause-specific death classification"].eq("Dead (missing/unknown COD)")
    | df_eligible["SEER other cause of death classification"].eq("Dead (missing/unknown COD)")
)
miss_tab["Unknown COD (% of deaths)"] = (
    ucod_eligible[dead_eligible].groupby(df_eligible.loc[dead_eligible, band_col]).mean() * 100
).round(2)
miss_tab.insert(0, "n_eligible", df_eligible.groupby(band_col).size())

print("Missing/unknown values by age band (eligible cohort, before covariate exclusions), %:")
display(miss_tab)

retention = (
    cohort.groupby(band_col).size().rename("n_analytic").to_frame()
    .join(df_eligible.groupby(band_col).size().rename("n_eligible"))
)
retention["retained_pct"] = (retention["n_analytic"] / retention["n_eligible"] * 100).round(1)
print("\nCascade retention by age band (analytic / eligible):")
display(retention[["n_eligible", "n_analytic", "retained_pct"]])

## Feature engineering (row-wise, before any split)

Every transformation below uses only the row's own values. Nothing is learned from the data, so it is safe to do before the CV split.

- **Coded counts → number + description.** Regional nodes examined/positive, LN positive axillary level I–II and tumor size are SEER *codes*, not plain numbers. For example, nodes-positive `97` means "positive, number unspecified", not 97 nodes. Each becomes:
  - a numeric column holding only true counts or sizes (`NaN` otherwise; `90` means "90 or more");
  - a `_desc` category holding the meaning of any special code.
- **Stage/TNM harmonization.** 2010–2015 (AJCC 7th derived) and 2016–2017 (SEER Combined) use different labels (`T1c` vs `p1C`). If left as is, the one-hot columns would reveal diagnosis year. Explicit maps put both eras on one vocabulary:
  - the clinical/pathologic `c`/`p` prefix is dropped (it does not exist before 2016);
  - bare parents (`N1`) and `NOS` labels merge into `_NOS`;
  - unmapped labels become `NaN`, which fails the QC check below.
- **Age.** `age_years` is continuous ("90+" → 90) and is the model input. `age_group_report` (<40, 40–49, 50–64, 65–79, 80+) is used only for descriptive tables and CV stratification.
- **County SES.**
  - `income_ord` is the lower bound of the income band ("< $40,000" → 39,999), so the ordering is kept.
  - `rural_urban_continuum` stays categorical, with an explicit "Unknown" level.
- **Outcome.** `event_cr` is coded 0 = censored (alive at last follow-up), 1 = breast-cancer death, 2 = other-cause death. It comes from SEER's cause-specific and other-cause death classifications. `event` = 1 for breast-cancer death.
- **Per-horizon labels** (spec §3.1), one set for each τ in `HORIZONS`:
  - `y_bc_τ` = 1 for a breast-cancer death at `time ≤ τ`, else 0. Other-cause deaths are 0, not censored.
  - `y_any_τ` = 1 for a death from any cause at `time ≤ τ`, for the all-cause sensitivity analysis.
  - `determinable_τ` = False if censored before τ.

  How the strategies use them: **Zero** uses all rows. **Discard** and **IPCW** use the determinable rows only. IPCW weights depend on Ĝ, so they are computed inside each fold, not here.
- **CV stratum.** `split_stratum` = `age_group_report` × `event_cr`.

In [ ]:
def extract_and_categorize_histology(data, source_col="ICD-O-3 Hist/behav, malignant"):
    data = data.copy()
    extracted_codes = data[source_col].astype(str).str.extract(r"^(\d{4})").astype(float)
    extracted_behav = data[source_col].astype(str).str.extract(r"/(\d)").fillna(3).astype(int)

    is_in_situ = extracted_behav[0] == 2
    is_invasive = extracted_behav[0] == 3
    c = extracted_codes[0]

    ductal_codes = [8500, 8501, 8502, 8504, 8507, 8523, 8541, 8543]
    lobular_codes = [8520, 8521, 8524, 8525]
    special_types = [8480, 8211, 8575, 8010, 8510, 8503, 8401, 8201, 8140, 8050, 8530, 8200, 8255]

    data.loc[is_in_situ & c.isin(ductal_codes), "Histology_Group"] = "DCIS (Ductal In Situ)"
    data.loc[is_in_situ & c.isin(lobular_codes), "Histology_Group"] = "LCIS (Lobular In Situ)"
    data.loc[is_in_situ & (c == 8522), "Histology_Group"] = "Mixed In Situ"
    data.loc[
        is_in_situ & ~c.isin(ductal_codes) & ~c.isin(lobular_codes) & (c != 8522),
        "Histology_Group",
    ] = "Other In Situ"

    data.loc[is_invasive & c.isin(ductal_codes), "Histology_Group"] = "Invasive Ductal"
    data.loc[is_invasive & c.isin(lobular_codes), "Histology_Group"] = "Invasive Lobular"
    data.loc[is_invasive & (c == 8522), "Histology_Group"] = "Mixed Ductal & Lobular"
    data.loc[is_invasive & c.isin(special_types), "Histology_Group"] = "Special Type Invasive"
    data["Histology_Group"] = data["Histology_Group"].fillna("Other / Unclassified")
    return data


def leading_number(series):
    """Numeric value of the leading digits ("000 (no evidence...)" -> 0); NaN if none."""
    return pd.to_numeric(series.astype(str).str.extract(r"^\s*(\d+)", expand=False), errors="coerce")


def add_raw_model_features(data):
    data = data.copy()

    data["Histologic_Type"] = data["Histologic Type ICD-O-3"].astype(str)
    data["Histology_broad_groupings"] = data["Histology recode - broad groupings"].astype(str)
    data = extract_and_categorize_histology(data)

    # ---- Age and county SES ----
    data["age_years"] = leading_number(data["Age recode with single ages and 90+"])  # "90+" -> 90
    data["age_band"] = data["Age recode with <1 year olds and 90+"].astype(str)
    data["age_group_report"] = pd.cut(
        data["age_years"],
        bins=[0, 40, 50, 65, 80, np.inf],
        right=False,
        labels=["<40", "40-49", "50-64", "65-79", "80+"],
    ).astype(str)

    income_raw = data["Median household income inflation adj to 2024"].astype(str).str.strip()
    income_low = pd.to_numeric(
        income_raw.str.replace(",", "", regex=False).str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )
    # "< $40,000" -> 39,999 so the open bottom band sorts below the first closed band
    data["income_ord"] = income_low.where(~income_raw.str.startswith("<"), income_low - 1)

    data["rural_urban_continuum"] = data["Rural-Urban Continuum Code"].astype(str).str.strip().replace("", "Unknown")

    # ---- Stage/TNM harmonization: both eras mapped onto one vocabulary ----
    # Labels missing from a map become NaN and fail the harmonization QC.
    STAGE_MAP_ROMAN = {   # 2010-2015 AJCC 7th ed and the 2016-2017 7th-ed recode
        "0": "0", "IA": "IA", "IB": "IB", "IIA": "IIA", "IIB": "IIB",
        "IIIA": "IIIA", "IIIB": "IIIB", "IIIC": "IIIC",
        "III": "III_NOS", "IIINOS": "III_NOS", "IV": "IV",
    }
    STAGE_MAP_CMB = {     # 2016-2017 Derived SEER Cmb Stg Grp (fallback)
        "0": "0", "1A": "IA", "1B": "IB", "2A": "IIA", "2B": "IIB",
        "3A": "IIIA", "3B": "IIIB", "3C": "IIIC", "3": "III_NOS", "4": "IV",
    }
    T_MAP_2010_2015 = {
        "T0": "T0", "Tis": "Tis", "T1mic": "T1mic",
        "T1a": "T1a", "T1b": "T1b", "T1c": "T1c", "T1": "T1_NOS", "T1NOS": "T1_NOS",
        "T2": "T2", "T3": "T3",
        "T4a": "T4a", "T4b": "T4b", "T4c": "T4c", "T4d": "T4d", "T4": "T4_NOS", "T4NOS": "T4_NOS",
    }
    T_MAP_2016_2017 = {   # keys after removing the c/p prefix and uppercasing
        "0": "T0", "IS": "Tis", "1MI": "T1mic",
        "1A": "T1a", "1B": "T1b", "1C": "T1c", "1": "T1_NOS",
        "2": "T2", "3": "T3",
        "4A": "T4a", "4B": "T4b", "4C": "T4c", "4D": "T4d", "4": "T4_NOS",
    }
    N_MAP_2010_2015 = {
        "N0": "N0", "N0(i-)": "N0(i-)", "N0(i+)": "N0(i+)",
        "N0(mol-)": "N0(mol-)", "N0(mol+)": "N0(mol+)",
        "N1mi": "N1mi", "N1a": "N1a", "N1b": "N1b", "N1c": "N1c",
        "N1": "N1_NOS", "N1NOS": "N1_NOS",
        "N2a": "N2a", "N2b": "N2b", "N2": "N2_NOS", "N2NOS": "N2_NOS",
        "N3a": "N3a", "N3b": "N3b", "N3c": "N3c", "N3": "N3_NOS", "N3NOS": "N3_NOS",
    }
    N_MAP_2016_2017 = {   # keys after removing the c/p prefix and uppercasing
        "0": "N0", "0I-": "N0(i-)", "0I+": "N0(i+)", "0M-": "N0(mol-)", "0M+": "N0(mol+)",
        "1MI": "N1mi", "1A": "N1a", "1B": "N1b", "1C": "N1c", "1": "N1_NOS",
        "2A": "N2a", "2B": "N2b", "2": "N2_NOS",
        "3A": "N3a", "3B": "N3b", "3C": "N3c", "3": "N3_NOS",
    }
    M_MAP_2010_2015 = {"M0": "M0", "M0(i+)": "M0(i+)", "M1": "M1"}
    M_MAP_2016_2017 = {"0": "M0", "0I+": "M0(i+)", "1": "M1"}

    era_1015 = pd.to_numeric(data["Year of diagnosis"], errors="coerce").between(2010, 2015)

    def _clean(series):
        return series.astype(str).str.strip()

    def _strip_cp_upper(series):
        return _clean(series).str.replace(r"^[cp]", "", regex=True).str.upper()

    stage_1015 = _clean(data["Derived AJCC Stage Group, 7th ed (2010-2015)"]).map(STAGE_MAP_ROMAN)
    stage_1617 = _clean(data["7th Edition Stage Group Recode (2016-2017)"]).map(STAGE_MAP_ROMAN)
    stage_1617 = stage_1617.fillna(_clean(data["Derived SEER Cmb Stg Grp (2016-2017)"]).map(STAGE_MAP_CMB))
    data["stage_group_harmonized"] = stage_1015.where(era_1015, stage_1617)

    t_1015 = _clean(data["Derived AJCC T, 7th ed (2010-2015)"]).map(T_MAP_2010_2015)
    t_1617 = _strip_cp_upper(data["Derived SEER Combined T (2016-2017)"]).map(T_MAP_2016_2017)
    data["t_stage_harmonized"] = t_1015.where(era_1015, t_1617)

    n_1015 = _clean(data["Derived AJCC N, 7th ed (2010-2015)"]).map(N_MAP_2010_2015)
    n_1617 = _strip_cp_upper(data["Derived SEER Combined N (2016-2017)"]).map(N_MAP_2016_2017)
    data["n_stage_harmonized"] = n_1015.where(era_1015, n_1617)

    m_1015 = _clean(data["Derived AJCC M, 7th ed (2010-2015)"]).map(M_MAP_2010_2015)
    m_1617 = _strip_cp_upper(data["Derived SEER Combined M (2016-2017)"]).map(M_MAP_2016_2017)
    data["m_stage_harmonized"] = m_1015.where(era_1015, m_1617)

    # ---- Time to treatment (treatment-inclusive feature set only) ----
    treatment_raw = data["Time from diagnosis to treatment in days recode"].replace({"731+ days": "731"})
    data["time_to_treatment_in_days_is_unknown"] = treatment_raw.isin(["Unable to calculate", "999", ""])
    data["time_to_treatment_in_days_raw"] = pd.to_numeric(treatment_raw, errors="coerce")
    data.loc[data["time_to_treatment_in_days_raw"].eq(999), "time_to_treatment_in_days_raw"] = np.nan

    # ---- Tumor size: mm for true measurements (001-400), else NaN + category ----
    size_code = leading_number(data["Tumor Size Over Time Recode (1988+)"])
    data["tumor_size_over_time"] = size_code.where(size_code.between(1, 400))
    data["tumor_size_desc"] = np.select(
        [size_code.between(1, 400), size_code.eq(0), size_code.eq(990), size_code.eq(998)],
        ["Exact size in mm", "No evidence of primary tumor", "Microscopic focus", "Site-specific code"],
        default="Unknown or size unreasonable",
    )

    # ---- Regional nodes examined: count for 00-90, procedure codes kept as categories ----
    exam_code = leading_number(data["Regional nodes examined (1988+)"])
    data["regional_nodes_examined_number"] = exam_code.where(exam_code.between(0, 90))
    data["regional_nodes_examined_desc"] = np.select(
        [
            exam_code.eq(0),
            exam_code.between(1, 89),
            exam_code.eq(90),
            exam_code.eq(95),
            exam_code.eq(96),
            exam_code.eq(97),
            exam_code.eq(98),
        ],
        [
            "No nodes were examined",
            "Exact number of nodes examined",
            "90 or more nodes were examined",
            "No regional nodes removed, aspiration performed",
            "Removal documented as sampling, number unknown",
            "Removal documented as dissection, number unknown",
            "Nodes surgically removed, number unknown",
        ],
        default="Unknown whether nodes were examined",   # 99 / blank / other
    )

    # ---- Regional nodes positive: same pattern ----
    pos_code = leading_number(data["Regional nodes positive (1988+)"])
    data["regional_nodes_positive_number"] = pos_code.where(pos_code.between(0, 90))
    data["regional_nodes_positive_desc"] = np.select(
        [
            pos_code.eq(0),
            pos_code.between(1, 89),
            pos_code.eq(90),
            pos_code.eq(95),
            pos_code.eq(97),
            pos_code.eq(98),
        ],
        [
            "All nodes examined are negative",
            "Exact number of nodes positive",
            "90 or more nodes are positive",
            "Positive aspiration of lymph node(s) was performed",
            "Positive nodes documented, number unspecified",
            "No nodes were examined",
        ],
        default="Unknown whether nodes are positive",   # 99 / blank / other
    )

    # ---- LN positive axillary level I-II: numbers and text labels are mixed ----
    axl_str = data["LN Positive Axillary Level I-II Recode (2010+)"].astype(str).str.strip()
    axl_lower = axl_str.str.lower()
    axl_num = leading_number(axl_str)
    is_negative_label = axl_lower.str.contains("negative", regex=False)  # code 00 exported as text
    data["ln_positive_axillary_number"] = axl_num.where(axl_num.between(0, 90)).mask(is_negative_label, 0.0)
    data["ln_positive_axillary_desc"] = np.select(
        [
            is_negative_label,
            axl_num.between(1, 89),
            axl_num.eq(90),
            axl_lower.str.contains("aspiration", regex=False),
            axl_lower.str.contains("unspecified", regex=False),
            axl_lower.str.contains("not examined", regex=False) | axl_lower.str.contains("no axillary", regex=False),
        ],
        [
            "All axillary level I-II nodes examined negative",
            "Exact number of positive axillary level I-II nodes",
            "90 or more positive axillary level I-II nodes",
            "Positive aspiration or needle core biopsy",
            "Positive nodes, number unspecified",
            "No axillary nodes examined",
        ],
        default="Unknown or not applicable",
    )

    # ---- Outcome: 0 = censored, 1 = breast-cancer death, 2 = other-cause death ----
    data["time"] = pd.to_numeric(data["Survival months"], errors="coerce").astype(float)
    data["event"] = data["SEER cause-specific death classification"].eq(
        "Dead (attributable to this cancer dx)"
    ).astype("int64")
    data["event_other"] = data["SEER other cause of death classification"].eq(
        "Dead (attributable to causes other than this cancer dx)"
    ).astype("int64")
    data["event_cr"] = np.select(
        [data["event"].eq(1), data["event_other"].eq(1)], [1, 2], default=0
    ).astype("int64")

    data["split_stratum"] = data["age_group_report"].astype(str) + "|" + data["event_cr"].astype(str)
    return data


def add_horizon_labels(data, horizons):
    """Per-horizon labels (spec §3.1). IPCW weights are fold-specific and are not built here."""
    data = data.copy()
    for tau in horizons:
        died_by_tau = data["event_cr"].gt(0) & data["time"].le(tau)
        data[f"y_bc_{tau}"] = (died_by_tau & data["event_cr"].eq(1)).astype("int64")
        data[f"y_any_{tau}"] = died_by_tau.astype("int64")
        data[f"determinable_{tau}"] = ~(data["event_cr"].eq(0) & data["time"].lt(tau))
    return data


LABEL_COLS = [f"{prefix}_{tau}" for tau in HORIZONS for prefix in ("y_bc", "y_any", "determinable")]

In [ ]:
model_df = add_horizon_labels(add_raw_model_features(cohort), HORIZONS)
print(model_df.shape)
model_df[
    ["time", "event_cr", "age_years", "age_group_report", "income_ord", "rural_urban_continuum",
     "tumor_size_over_time", "tumor_size_desc", "regional_nodes_positive_number"] + LABEL_COLS
].head()

### QC: number and description columns agree

A numeric column must be filled **exactly** where its description is a true count or measured size, and `NaN` everywhere else. The tables show record counts and the numeric range per category. The assertions stop the notebook if a new SEER label breaks the rule.

In [ ]:
qc_specs = [
    ("regional_nodes_examined_number", "regional_nodes_examined_desc",
     {"No nodes were examined", "Exact number of nodes examined", "90 or more nodes were examined"}),
    ("regional_nodes_positive_number", "regional_nodes_positive_desc",
     {"All nodes examined are negative", "Exact number of nodes positive", "90 or more nodes are positive"}),
    ("tumor_size_over_time", "tumor_size_desc",
     {"Exact size in mm"}),
    ("ln_positive_axillary_number", "ln_positive_axillary_desc",
     {"All axillary level I-II nodes examined negative",
      "Exact number of positive axillary level I-II nodes",
      "90 or more positive axillary level I-II nodes"}),
]

for num_col, desc_col, count_categories in qc_specs:
    print(f"=== {desc_col} vs {num_col} ===")
    summary = (
        model_df.groupby(desc_col, dropna=False)[num_col]
        .agg(n="size", n_numeric="count", min="min", max="max")
        .sort_values("n", ascending=False)
    )
    display(summary)

    in_count_category = model_df[desc_col].isin(count_categories)
    bad_numeric = model_df[~in_count_category & model_df[num_col].notna()]
    assert bad_numeric.empty, (
        f"{num_col} has values in non-count categories: {bad_numeric[desc_col].value_counts().to_dict()}"
    )
    assert not (in_count_category & model_df[num_col].isna()).any(), f"{num_col} is NaN inside count categories"

print("QC passed: numeric columns are filled exactly where the description is a true count/size.")

### QC: age and income

1. `age_years` must parse for every row and fall inside the row's own 5-year age band. The two age columns were exported separately, so agreement rules out a shift or parsing error.
2. The income label → `income_ord` mapping must be one-to-one in both directions: no two bands merged, no band split.
3. `age_group_report` must be assigned for every row.

In [ ]:
assert model_df["age_years"].notna().all(), "age_years failed to parse for some rows"


def _band_bounds(label):
    nums = re.findall(r"\d+", str(label))
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    if len(nums) == 1:  # "90+ years"
        return float(nums[0]), np.inf
    return np.nan, np.nan


_bounds = model_df["age_band"].map(_band_bounds)
_low = _bounds.str[0].astype(float)
_high = _bounds.str[1].astype(float)
bad_age = model_df[(model_df["age_years"] < _low) | (model_df["age_years"] > _high)]
assert bad_age.empty, f"{len(bad_age):,} rows where age_years falls outside its band:\n{bad_age[['age_years', 'age_band']].head()}"

income_col = "Median household income inflation adj to 2024"
known_income = model_df["income_ord"].notna()
label_to_ord = model_df.loc[known_income].groupby(income_col)["income_ord"].nunique()
ord_to_label = model_df.loc[known_income].groupby("income_ord")[income_col].nunique()
assert (label_to_ord == 1).all(), "an income label maps to several ordinal values"
assert (ord_to_label == 1).all(), f"income labels collide on one ordinal value:\n{ord_to_label[ord_to_label > 1]}"
print(f"Income bands mapped: {label_to_ord.size} | unknown income: {(~known_income).mean():.2%}")

assert model_df["age_group_report"].ne("nan").all(), "age_group_report has unassigned rows"
print("Age and income QC passed.")

### QC: stage/TNM harmonization

1. No row may have `NaN` in a harmonized column; `NaN` means a source label is missing from the maps (hard check).
2. Every level should appear in both eras. A level with zero count in one era would reveal the era through its one-hot column, so the cell warns if one is found.

In [ ]:
harmonized_cols = ["stage_group_harmonized", "t_stage_harmonized", "n_stage_harmonized", "m_stage_harmonized"]
harmonized_sources = {
    "stage_group_harmonized": [
        "Derived AJCC Stage Group, 7th ed (2010-2015)",
        "7th Edition Stage Group Recode (2016-2017)",
        "Derived SEER Cmb Stg Grp (2016-2017)",
    ],
    "t_stage_harmonized": ["Derived AJCC T, 7th ed (2010-2015)", "Derived SEER Combined T (2016-2017)"],
    "n_stage_harmonized": ["Derived AJCC N, 7th ed (2010-2015)", "Derived SEER Combined N (2016-2017)"],
    "m_stage_harmonized": ["Derived AJCC M, 7th ed (2010-2015)", "Derived SEER Combined M (2016-2017)"],
}

for col in harmonized_cols:
    bad = model_df[model_df[col].isna()]
    if not bad.empty:
        print(f"UNMAPPED SOURCE LABELS for {col}:")
        display(bad[harmonized_sources[col] + ["Year of diagnosis"]].value_counts().head(25))
    assert bad.empty, f"{col}: {len(bad):,} rows carry labels missing from the harmonization maps"
print("Check 1 passed: every row maps into the shared vocabulary.\n")

era_label = (
    pd.to_numeric(model_df["Year of diagnosis"], errors="coerce")
    .ge(2016)
    .map({False: "2010-2015", True: "2016-2017"})
)
print(f"Overall 2016-2017 share of the cohort: {(era_label == '2016-2017').mean():.3f}\n")

for col in harmonized_cols:
    ct = pd.crosstab(model_df[col], era_label)
    for era_col in ["2010-2015", "2016-2017"]:
        if era_col not in ct.columns:
            ct[era_col] = 0
    ct["share_2016_2017"] = (ct["2016-2017"] / ct.sum(axis=1)).round(3)
    print(f"=== {col} ===")
    display(ct.sort_values("2010-2015", ascending=False))
    one_era_only = ct[(ct["2010-2015"] == 0) | (ct["2016-2017"] == 0)]
    if not one_era_only.empty:
        print(f"WARNING: levels present in only one era: {list(one_era_only.index)}\n")

## Outcome validation

### Coding checks

These hard checks guard against a mistyped label string silently producing a wrong target.
- Only the two expected labels remain in each death classification column.
- A patient is never coded as both a breast-cancer death and an other-cause death.
- `event_cr > 0` exactly when vital status is Dead.
- All survival times are positive.
- The per-horizon labels are consistent: a breast-cancer death is also an all-cause death, and every death by τ is determinable.

In [ ]:
cs = model_df["SEER cause-specific death classification"]
oth = model_df["SEER other cause of death classification"]
vital = model_df["Vital status recode (study cutoff used)"]

ALLOWED_CS = {"Alive or dead of other cause", "Dead (attributable to this cancer dx)"}
ALLOWED_OTH = {"Alive or dead due to cancer", "Dead (attributable to causes other than this cancer dx)"}

unexpected_cs = set(cs.unique()) - ALLOWED_CS
assert not unexpected_cs, f"Unexpected cause-specific labels: {unexpected_cs}"
unexpected_oth = set(oth.unique()) - ALLOWED_OTH
assert not unexpected_oth, f"Unexpected other-cause labels: {unexpected_oth}"

overlap = ((model_df["event"] == 1) & (model_df["event_other"] == 1)).sum()
assert overlap == 0, f"{overlap} rows coded as both breast-cancer and other-cause death"
assert (model_df["event"] == (model_df["event_cr"] == 1).astype(int)).all()

mismatch = (vital.eq("Dead") != model_df["event_cr"].gt(0)).sum()
assert mismatch == 0, f"{mismatch} rows where event_cr disagrees with vital status"

assert model_df["time"].notna().all() and (model_df["time"] > 0).all()

for tau in HORIZONS:
    assert (model_df[f"y_bc_{tau}"] <= model_df[f"y_any_{tau}"]).all(), f"y_bc_{tau} > y_any_{tau}"
    assert model_df.loc[model_df[f"y_any_{tau}"] == 1, f"determinable_{tau}"].all(), f"death by {tau} not determinable"

print("Outcome coding checks passed.")

### Event counts and Kaplan–Meier check

The breast-cancer-specific KM treats other-cause deaths as censored, so 1 − KM here is the *net* risk, not the study target. Cross-check S(τ) against a SEER\*Stat cause-specific survival session that applies this notebook's exclusions. Median follow-up is estimated with reverse KM.

In [ ]:
def km_survival(durations, events):
    """Product-limit estimator. Returns (event_times, S(t) at those times)."""
    d = np.asarray(durations, dtype=float)
    e = np.asarray(events, dtype=int)
    order = np.argsort(d, kind="mergesort")
    d, e = d[order], e[order]
    event_times, deaths = np.unique(d[e == 1], return_counts=True)
    at_risk = len(d) - np.searchsorted(d, event_times, side="left")
    surv = np.cumprod(1.0 - deaths / at_risk)
    return event_times, surv


def km_at(times, surv, t):
    idx = np.searchsorted(times, t, side="right") - 1
    return 1.0 if idx < 0 else float(surv[idx])

In [ ]:
t_arr = model_df["time"].to_numpy()
e_arr = model_df["event"].to_numpy()
ecr = model_df["event_cr"].to_numpy()
n = len(model_df)

print(f"Analytic cohort       : {n:,}")
for code_value, label in [(1, "Breast-cancer deaths"), (2, "Other-cause deaths"), (0, "Censored (alive)")]:
    k = int((ecr == code_value).sum())
    print(f"{label:<22}: {k:,} ({k / n:.2%})")
print(f"Observed time (months): min {t_arr.min():.0f}, median {np.median(t_arr):.0f}, max {t_arr.max():.0f}")

km_t, km_s = km_survival(t_arr, e_arr)
km_at_tau = {tau: km_at(km_t, km_s, tau) for tau in HORIZONS}
print()
for tau, s in km_at_tau.items():
    print(f"Breast-cancer-specific survival at {tau:>3} months (KM): {s:.4f}")

rk_t, rk_s = km_survival(t_arr, (ecr == 0).astype(int))
below = np.nonzero(rk_s <= 0.5)[0]
median_fu = float(rk_t[below[0]]) if len(below) else float("nan")
print(f"\nMedian follow-up (reverse KM): {median_fu:.0f} months")

fig, ax = plt.subplots(figsize=(8, 5))
ax.step(np.concatenate([[0.0], km_t]), np.concatenate([[1.0], km_s]), where="post", lw=1.8)
for tau, color in zip(HORIZONS, ["seagreen", "crimson", "darkorange"]):
    ax.axvline(tau, color=color, ls="--", lw=1.2, label=f"{tau} months: S = {km_at_tau[tau]:.3f}")
ax.set_xlabel("Months since diagnosis")
ax.set_ylabel("Breast-cancer-specific survival")
ax.set_title("Kaplan-Meier: breast-cancer-specific survival (analytic cohort)")
ax.set_ylim(0.80, 1.005)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Table T1 — cohort by horizon

For each horizon, the table shows:
- total and determinable cases;
- deaths by cause, and censored before τ;
- the imbalance ratio (non-events / breast-cancer events) under **Zero** (all rows) and under **Discard/IPCW** (determinable rows).

Other-cause deaths before τ are the patients that a "censor other-cause deaths" rule would have discarded (spec §8.5.3). Effective sample size depends on Ĝ, so it is reported in the modeling notebook.

In [ ]:
t1_rows = []
for tau in HORIZONS:
    y_bc = model_df[f"y_bc_{tau}"]
    det = model_df[f"determinable_{tau}"]
    n_bc = int(y_bc.sum())
    t1_rows.append({
        "horizon_months": tau,
        "n_total": n,
        "n_determinable": int(det.sum()),
        "bc_deaths": n_bc,
        "other_cause_deaths": int((model_df["event_cr"].eq(2) & model_df["time"].le(tau)).sum()),
        "censored_before_tau": int((~det).sum()),
        "at_risk_at_tau": int(model_df["time"].ge(tau).sum()),
        "IR_zero": round((n - n_bc) / n_bc, 2),
        "IR_discard_ipcw": round((int(det.sum()) - n_bc) / n_bc, 2),
    })
table_t1 = pd.DataFrame(t1_rows).set_index("horizon_months")
display(table_t1)
table_t1.to_csv(OUTPUT_DIR / "table_T1_by_horizon.csv")

### Aalen–Johansen check and table S3 (CIF vs net risk)

Aalen–Johansen estimates CIF₁(τ) = P(T ≤ τ, breast-cancer death), the quantity the labels target. The identity CIF₁ + CIF₂ + S = 1 is asserted.

Table S3 compares CIF₁ with the net risk 1 − KM, by horizon and age group. The gap shows how much treating other-cause deaths as censored overstates breast-cancer risk (spec §8.5.2). Cross-check CIF₁ and CIF₂ against a SEER\*Stat cumulative-incidence session.

In [ ]:
def aalen_johansen(durations, event_cr, causes=(1, 2)):
    """Aalen-Johansen cumulative incidence.

    event_cr: 0 censored, 1 / 2 = cause. Returns (death_times, {cause: CIF}, S_allcause).
    """
    d = np.asarray(durations, dtype=float)
    e = np.asarray(event_cr, dtype=int)
    order = np.argsort(d, kind="mergesort")
    d, e = d[order], e[order]

    death_times = np.unique(d[e > 0])
    n_at_risk = len(d) - np.searchsorted(d, death_times, side="left")
    d_all = np.array([((d == t) & (e > 0)).sum() for t in death_times])
    s_all = np.cumprod(1.0 - d_all / n_at_risk)       # S(t) just after each death time
    s_left = np.concatenate([[1.0], s_all[:-1]])      # S(t-)

    cifs = {}
    for k in causes:
        d_k = np.array([((d == t) & (e == k)).sum() for t in death_times])
        cifs[k] = np.cumsum(s_left * d_k / n_at_risk)
    return death_times, cifs, s_all


def cif_at(times, cif, t):
    idx = np.searchsorted(times, t, side="right") - 1
    return 0.0 if idx < 0 else float(cif[idx])

In [ ]:
aj_t, aj_cif, aj_s = aalen_johansen(t_arr, ecr)
identity_gap = np.abs(aj_cif[1] + aj_cif[2] + aj_s - 1.0).max()
assert identity_gap < 1e-9, f"AJ identity violated (max |CIF1+CIF2+S-1| = {identity_gap:.2e})"
print(f"AJ identity check passed (max |CIF1+CIF2+S-1| = {identity_gap:.1e})")

AGE_GROUPS = ["<40", "40-49", "50-64", "65-79", "80+"]
groups = [("All", np.ones(n, dtype=bool))] + [
    (g, (model_df["age_group_report"] == g).to_numpy()) for g in AGE_GROUPS
]

s3_rows = []
for group, m in groups:
    km_tg, km_sg = km_survival(t_arr[m], e_arr[m])
    aj_tg, aj_cg, _ = aalen_johansen(t_arr[m], ecr[m])
    for tau in HORIZONS:
        net = 1.0 - km_at(km_tg, km_sg, tau)
        cif1 = cif_at(aj_tg, aj_cg[1], tau)
        s3_rows.append({
            "horizon_months": tau,
            "age_group": group,
            "n": int(m.sum()),
            "net_risk_1_minus_KM": round(net, 4),
            "CIF1_breast_cancer": round(cif1, 4),
            "gap_pp": round((net - cif1) * 100, 2),
            "CIF2_other_cause": round(cif_at(aj_tg, aj_cg[2], tau), 4),
        })

table_s3 = pd.DataFrame(s3_rows).set_index(["horizon_months", "age_group"]).sort_index(level=0, sort_remaining=False)
display(table_s3)
table_s3.to_csv(OUTPUT_DIR / "table_S3_cif_vs_net.csv")

## Feature sets

- **`diagnosis_only`** (primary) uses information available at diagnosis only.
- **`treatment_inclusive`** (sensitivity only) adds treatment variables. Chemotherapy is under-recorded in SEER, and time to treatment is measured after diagnosis.

`METADATA_COLS` lists the columns that must never enter a feature matrix: identifiers, strata, outcome and labels. The cell asserts that the feature sets do not overlap with it. `Year of diagnosis` is metadata: it enters the censoring model Ĝ only, never the outcome model (spec §3.2).

In [ ]:
diagnosis_numeric_cols = [
    "age_years",
    "income_ord",
    "tumor_size_over_time",
    "regional_nodes_examined_number",
    "regional_nodes_positive_number",
    "ln_positive_axillary_number",
]

diagnosis_log_numeric_cols = []

diagnosis_categorical_cols = [
    RACE_COL,
    "Marital status at diagnosis",
    "Primary Site - labeled",
    "Primary Site",
    "Laterality",
    "Grade Recode (thru 2017)",
    "Histologic_Type",
    "Histology_broad_groupings",
    "Histology_Group",
    "stage_group_harmonized",
    "t_stage_harmonized",
    "n_stage_harmonized",
    "m_stage_harmonized",
    "Breast Subtype (2010+)",
    "ER Status Recode Breast Cancer (1990+)",
    "PR Status Recode Breast Cancer (1990+)",
    "Derived HER2 Recode (2010+)",
    "ln_positive_axillary_desc",
    "regional_nodes_examined_desc",
    "regional_nodes_positive_desc",
    "tumor_size_desc",
    "SEER Combined Mets at DX-bone (2010+)",
    "SEER Combined Mets at DX-brain (2010+)",
    "SEER Combined Mets at DX-liver (2010+)",
    "SEER Combined Mets at DX-lung (2010+)",
    "rural_urban_continuum",
]

treatment_numeric_cols = [
    "time_to_treatment_in_days_is_unknown",
    "radiotherapy_uncertain",
    "systemic_sequence_unknown",
    "chemo_no_unknown",
    "treatment_uncertain_any",
]
treatment_log_numeric_cols = ["time_to_treatment_in_days_raw"]
treatment_categorical_cols = [
    "RX Summ--Surg Prim Site (1998-2022)",
    "RX Summ--Surg/Rad Seq",
    "Radiation recode",
    "Chemotherapy recode (yes, no/unk)",
    "RX Summ--Systemic/Sur Seq (2007+)",
]

feature_sets = {
    "diagnosis_only": {
        "numeric": diagnosis_numeric_cols,
        "log_numeric": diagnosis_log_numeric_cols,
        "categorical": diagnosis_categorical_cols,
    },
    "treatment_inclusive": {
        "numeric": diagnosis_numeric_cols + treatment_numeric_cols,
        "log_numeric": diagnosis_log_numeric_cols + treatment_log_numeric_cols,
        "categorical": diagnosis_categorical_cols + treatment_categorical_cols,
    },
}

METADATA_COLS = [
    "Patient ID",
    "Sequence number",
    "Type of Reporting Source",
    "Total number of in situ/malignant tumors for patient",
    "Year of diagnosis",
    "age_band",
    "age_group_report",
    "split_stratum",
    "time",
    "event",
    "event_other",
    "event_cr",
] + LABEL_COLS

for name, spec in feature_sets.items():
    cols = set(spec["numeric"] + spec["log_numeric"] + spec["categorical"])
    missing = sorted(cols - set(model_df.columns))
    assert not missing, f"{name} has missing feature columns: {missing}"
    leaked = cols & set(METADATA_COLS)
    assert not leaked, f"{name}: metadata columns in the features: {leaked}"
print("Feature sets valid; metadata columns are disjoint from all feature lists.")

feature_sets

## Era-leak check

Year of diagnosis must not reach the outcome model (spec pitfall 9). A logistic regression tries to predict the era (diagnosed ≥ 2016) from the diagnosis-only features, using a 60,000-row sample. Levels below 0.5% are pooled first, as in the modeling encoder; keep the two thresholds in sync.

How to read the AUC:
- **≲ 0.60:** pass.
- **0.60–0.75:** inspect the top coefficients.
- **> 0.75:** structural leak; fix before modeling.

County income and rural–urban code are linked by diagnosis year, so some genuine drift through them is expected. Report the AUC in Methods.

In [ ]:
leak_spec = feature_sets["diagnosis_only"]
leak_num_cols = leak_spec["numeric"] + leak_spec["log_numeric"]
leak_cat_cols = leak_spec["categorical"]

y_era = pd.to_numeric(model_df["Year of diagnosis"], errors="coerce").ge(2016).astype(int)

N_PROBE = 60_000
probe_idx = model_df.sample(n=N_PROBE, random_state=RANDOM_STATE).index if len(model_df) > N_PROBE else model_df.index

X_num = model_df.loc[probe_idx, leak_num_cols].apply(pd.to_numeric, errors="coerce")
X_num = X_num.fillna(X_num.median())
X_num = (X_num - X_num.mean()) / X_num.std(ddof=0).replace(0, 1)

RARE_LEVEL_MIN_FREQ = 0.005  # keep in sync with the modeling encoder's min_frequency
X_cat_raw = model_df.loc[probe_idx, leak_cat_cols].astype(str)
for col in X_cat_raw.columns:
    freq = X_cat_raw[col].value_counts(normalize=True)
    rare = set(freq.index[freq < RARE_LEVEL_MIN_FREQ])
    if rare:
        X_cat_raw[col] = X_cat_raw[col].where(~X_cat_raw[col].isin(rare), "__rare__")
X_cat = pd.get_dummies(X_cat_raw, dummy_na=False)
X_probe = pd.concat([X_num, X_cat], axis=1).astype("float32")
y_probe = y_era.loc[probe_idx]

print(f"Probe sample: {len(X_probe):,} rows, {X_probe.shape[1]} encoded features")
print(f"Era base rate (2016-2017): {y_probe.mean():.3f}")

In [ ]:
clf = LogisticRegression(max_iter=3000)
aucs = cross_val_score(clf, X_probe, y_probe, cv=3, scoring="roc_auc", n_jobs=-1)
auc_mean = aucs.mean()
print(f"\nEra-prediction AUC (3-fold CV): {auc_mean:.3f}  (folds: {np.round(aucs, 3)})")

clf.fit(X_probe, y_probe)
top_coefs = pd.Series(clf.coef_[0], index=X_probe.columns).abs().sort_values(ascending=False).head(15)
print("\nTop-|coefficient| features:")
display(top_coefs)

if auc_mean > 0.75:
    print("FAIL: structural era leak -- a feature separates the eras nearly deterministically.")
elif auc_mean > 0.60:
    print("BORDERLINE: inspect the top coefficients above before proceeding.")
else:
    print("PASS: only mild era signal remains, consistent with genuine case-mix drift.")

## Save model-ready outputs

In [ ]:
display(pd.DataFrame(exclusion_log))
print(model_df.shape)

In [ ]:
pd.DataFrame(exclusion_log).to_csv(OUTPUT_DIR / "exclusion_log.csv", index=False)
model_df.to_csv(OUTPUT_DIR / "model_df.csv", index=False)

metadata = {
    "cohort_version": "v4_2026-09_one_primary_only_ages20plus",
    "data_source": "SEER 17, November 2025 submission (Session 6 extract), follow-up to 31 Dec 2023",
    "cohort": "One primary only; diagnosed 2010-2017; ages 20-90+",
    "random_state": RANDOM_STATE,
    "horizons_months": HORIZONS,
    "estimand": "F1(tau | X) = P(T <= tau, breast-cancer death | X), cumulative incidence of breast-cancer death",
    "event_cr_definition": {
        "0": "censored (alive at last follow-up)",
        "1": "breast-cancer death (SEER cause-specific death classification)",
        "2": "other-cause death (SEER other cause of death classification)",
    },
    "label_columns": {
        str(tau): {"y_bc": f"y_bc_{tau}", "y_any": f"y_any_{tau}", "determinable": f"determinable_{tau}"}
        for tau in HORIZONS
    },
    "label_rules": {
        "y_bc_tau": "1 if breast-cancer death at time <= tau, else 0 (other-cause deaths are 0)",
        "y_any_tau": "1 if death from any cause at time <= tau (all-cause sensitivity analysis)",
        "determinable_tau": "False if censored (event_cr == 0) with time < tau",
        "zero_strategy": "all rows, label y_bc_tau",
        "discard_ipcw_strategies": "determinable_tau rows only; IPCW weights computed inside each fold",
    },
    "censoring_model": {
        "event": "censoring (event_cr == 0); both death types treated as censored",
        "covariates": "the 78-column design matrix of the 28 predictors (see prep_manifest.json); year of diagnosis is not used",
        "fit": "on the outer training set of each split (seer_modeling.py)",
    },
    "feature_sets": feature_sets,
    "metadata_cols": METADATA_COLS,
    "n_dataset": int(len(model_df)),
    "n_event_bc": int((model_df["event_cr"] == 1).sum()),
    "n_event_other": int((model_df["event_cr"] == 2).sum()),
    "n_censored": int((model_df["event_cr"] == 0).sum()),
    "age_reporting_strata": AGE_GROUPS,
    "axillary_node_policy": (
        "Node counts retained; unknown/procedure codes are NaN in *_number with the meaning kept in *_desc. "
        "Sensitivity analysis: restrict to rows whose *_desc is a true-count category."
    ),
    "split_instruction": (
        "Stratified 5-fold CV (repeated or 5x2 recommended) on split_stratum (age_group_report x event_cr). "
        "Hold out an inner validation partition from each training fold for threshold selection."
    ),
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Outputs saved to: {OUTPUT_DIR}")